# Chapter 3 · Schemas and contracts

An audit of the schedule of Northside Veterinary Clinic finds appointments that should not exist: on a Saturday, at 3 p.m., in the past. The incident is reproduced with twelve booking requests, seven of them with a trap, and four versions of `book_appointment` are tried: without validation, with a more precise schema, with Pydantic's types and with a full contract that applies the clinic's rules and explains its errors.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic

The clinic, its tools, the model and the agent are those of chapter 1, toolset 4.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0 pydantic==2.14.0  # the versions the book was checked with

In [ ]:
"""Chapter 3: schemas and contracts. An audit of the schedule finds appointments that
should not exist; we investigate why and fix it with schemas and validation."""
import copy
import datetime
import json
import re
import unicodedata
import urllib.request
from datetime import date
from pathlib import Path

import torch
from pydantic import BaseModel, Field, ValidationError, field_validator
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without diacritics, to compare names written in any form."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in text if not unicodedata.combining(c)).strip()


def book_appointment(pet, vet, date, time, reason):
    """Books a free slot in the schedule; returns the appointment or the problem."""
    if pet not in PETS:
        return {"error": f"There is no pet {pet}."}
    if vet not in VETS:
        return {"error": f"There is no vet {vet}."}
    if any(c["date"] == date and c["time"] == time and c["vet"] == vet
           and c["status"] != "canceled" for c in APPOINTMENTS):
        return {"error": f"{VETS[vet]} already has an appointment on {date} "
                         f"at {time}."}
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": vet, "pet": pet,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return appointment


def cancel_appointment(appointment):
    """Cancels a scheduled appointment."""
    for c in APPOINTMENTS:
        if c["id"] == appointment:
            if c["status"] != "scheduled":
                return {"error": f"Appointment {appointment} is \"{c['status']}\"."}
            c["status"] = "canceled"
            return {"canceled": appointment, "date": c["date"], "time": c["time"]}
    return {"error": f"There is no appointment {appointment}."}


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}


def age(birth_date):
    """Years completed on the date of the data."""
    n = date.fromisoformat(birth_date)
    return TODAY.year - n.year - ((TODAY.month, TODAY.day) < (n.month, n.day))


def pet_summary(pid):
    """The essentials of a pet, without the full medical history."""
    m = PETS[pid]
    return {"id": pid, "name": m["name"], "species": m["species"], "breed": m["breed"],
            "age_years": age(m["birth_date"]), "current_weight_kg": m["weight_kg"],
            "allergies": m["allergies"] or "none known",
            "owner": {"id": m["owner"], "name": OWNERS[m["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def client_record_3(client_id):
    """The client's record, summarized, with the total they owe."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [f for f in INVOICES if f["owner"] == client_id and not f["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": f["id"], "date": f["date"], "total": f["total"]}
                                for f in unpaid],
            "total_unpaid": round(sum(f["total"] for f in unpaid), 2)}


def schedule_3(date):
    """The day's appointments, one per line, and each vet's free times."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date)):
        return {"error": f"The date \"{date}\" is not valid; use YYYY-MM-DD."}
    appts = [c for c in APPOINTMENTS if c["date"] == date and c["status"] != "canceled"]
    return {"date": date, "appointments": [f"{c['id']} {c['time']} {c['vet']} "
                                           f"{PETS[c['pet']]['name']} ({c['pet']})"
                                           for c in appts],
            "vets": VETS,
            "free_times": {v: [t for t in TIMES if not any(
                c["time"] == t and c["vet"] == v for c in appts)]
                for v in VETS}}


def locate(pet, owner=""):
    """The pet's id from its name (and the owner's, if there are several)."""
    ids = [m for m, d in PETS.items() if strip_accents(pet) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        return None, {"error": f"There is no pet called \"{pet}\""
                               + (f" owned by {owner}" if owner else "")
                               + " at the clinic."}
    if len(ids) > 1:
        return None, {"error": f"There are {len(ids)} pets called \"{pet}\"; give "
                               "the owner's name.",
                      "pets": [pet_summary(m) for m in ids]}
    return ids[0], None


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def find_pet(name, owner=""):
    """A pet's details, looked up by name."""
    pid, error = locate(name, owner)
    return error or pet_summary(pid)


def pets_of(owner):
    """The pets of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{"name": OWNERS[d]["name"], "pets": [
        {"name": x["name"], "species": x["species"], "breed": x["breed"]}
        for m, x in PETS.items() if x["owner"] == d]} for d in ids]}


def vaccines(pet, owner=""):
    """A pet's vaccines, each with its next date and status."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "vaccines": vaccines_with_status(pid)}


def history(pet, last=3, owner=""):
    """A pet's last visits."""
    pid, error = locate(pet, owner)
    if error:
        return error
    return {"pet": PETS[pid]["name"], "visits": [
        {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
        for v in HISTORY[pid][-int(last):]]}


def dose(medicine, pet, owner=""):
    """The dose of a medicine for a pet, with allergy and species warnings."""
    med = PHARMACY.get(strip_accents(medicine))
    if med is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    pid, error = locate(pet, owner)
    if error:
        return error
    m = PETS[pid]
    warnings = []
    if strip_accents(medicine) in [strip_accents(a) for a in m["allergies"]]:
        warnings.append(f"{m['name']} is allergic to {medicine}: do not prescribe.")
    if m["species"] not in med["species"]:
        warnings.append(f"{medicine} is not indicated for {m['species']}s.")
    return {"medicine": strip_accents(medicine), "pet": m["name"],
            "weight_kg": m["weight_kg"], "dose_mg_per_kg": med["dose_mg_kg"],
            "dose_mg_per_dose": round(med["dose_mg_kg"] * m["weight_kg"], 2),
            "stock": med["stock"],
            "restock": med.get("restock", "no date") if not med["stock"] else None,
            "warnings": warnings or "none"}


def free_slots(date, vet=""):
    """The free times of a day, for one vet or for all of them."""
    r = schedule_3(date)
    if "error" in r:
        return r
    vets = [vet_of(vet)] if vet else list(VETS)
    if vets == [None]:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    return {"date": date, "free_times": {VETS[v]: r["free_times"][v] for v in vets}}


def appointments_of(pet, owner=""):
    """A pet's appointments, past and upcoming, with their id."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "appointments": [
        {"id": c["id"], "date": c["date"], "time": c["time"], "reason": c["reason"],
         "vet": VETS[c["vet"]], "status": c["status"]}
        for c in APPOINTMENTS if c["pet"] == pid]}


def accounts(owner):
    """The unpaid invoices of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{k: v for k, v in client_record_3(d).items() if k != "id"}
                        for d in ids]}


def book(pet, vet, date, time, reason, owner=""):
    """Books an appointment from the names of the pet and the vet."""
    pid, error = locate(pet, owner)
    if error:
        return error
    v = vet_of(vet)
    if v is None:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    r = book_appointment(pid, v, date, time, reason)
    if "error" in r:
        return r
    return {"booked": r["id"], "pet": PETS[pid]["name"],
            "vet": VETS[v], "date": date, "time": time}


def notify_owner_of(pet, text, owner=""):
    """Sends a notice to the owner of the pet."""
    pid, error = locate(pet, owner)
    return error or notify(PETS[pid]["owner"], text)


TASKS_SET = {"find_pet": find_pet, "pets_of": pets_of,
             "vaccines": vaccines, "history": history, "dose": dose,
             "free_slots": free_slots, "appointments_of": appointments_of,
             "accounts": accounts, "book_appointment": book,
             "cancel_appointment": cancel_appointment, "notify_owner": notify_owner_of}

In [ ]:
def tool(name, description, parameters):
    """Schema of a tool; the parameters without a default value are required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "required": [p for p, (_, req) in parameters.items() if req],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parameters.items()}}}}


PET = ("Name of the pet.", True)


OWNER = ("Owner's name; only if several pets have that name.", False)


SCHEMAS_TASKS = [
    tool("find_pet", "Details of a pet looked up by its name: species, age, current "
         "weight, allergies and owner.", {"name": PET, "owner": OWNER}),
    tool("pets_of", "The pets of a client, looked up by name.",
         {"owner": ("Owner's name.", True)}),
    tool("vaccines", "Vaccines of a pet: the next date of each one and whether it is "
         "overdue.", {"pet": PET, "owner": OWNER}),
    tool("history", "The last visits of a pet: date, reason, treatment and weight.",
         {"pet": PET, "last": ("How many visits (3 by default).", False),
          "owner": OWNER}),
    tool("dose", "Dose of a medicine for a pet according to its current weight, with "
         "allergy or species warnings and the stock.",
         {"medicine": ("Name of the medicine.", True), "pet": PET, "owner": OWNER}),
    tool("free_slots", "Free times in the schedule on a day, for one vet or for all "
         "of them.", {"date": ("YYYY-MM-DD.", True),
                      "vet": ("Name of the vet; empty for all.", False)}),
    tool("appointments_of", "Appointments of a pet (past and upcoming), with their id.",
         {"pet": PET, "owner": OWNER}),
    tool("accounts", "Unpaid invoices of a client, looked up by name, and the total "
         "they owe.", {"owner": ("Owner's name.", True)}),
    tool("book_appointment", "Books an appointment at a free time.",
         {"pet": PET, "vet": ("Name of the vet.", True),
          "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True), "owner": OWNER}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Id of the appointment, such as AP-0026 (given by "
                          "appointments_of).", True)}),
    tool("notify_owner", "Sends a notice to the owner of a pet, by email or SMS as "
         "they prefer.", {"pet": PET, "text": ("Text of the notice, in English.", True),
                          "owner": OWNER}),
]


SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer of the model and its input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def execute(call, functions):
    """Executes a call if the tool exists and the arguments fit."""
    function = functions.get(call["name"])
    if function is None:
        return {"error": f"There is no tool {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Invalid arguments: {e}"}


def agent(task, functions, schemas, max_steps=8, show=False):
    """Think, act, observe and repeat. Counts the tokens of the results."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, schemas)
        calls = read_calls(answer)[:1]                   # one at a time
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "output_tokens": n_out,
                      "thought": thought, "calls": []})
        if not calls:
            trace[-1]["answer"] = answer
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        if show and thought:
            print(f"[{step}] {thought}")
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = execute(call, functions)
            text = json.dumps(result, ensure_ascii=False)
            trace[-1]["calls"].append({**call, "result": result,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def new_appointments():
    """The appointments created during the task."""
    return [c for c in APPOINTMENTS if c["id"] not in {x["id"] for x in APPOINTMENTS0}]

## 2. The audit

The rules of an appointment, written in Python. With chapter 1's tool, a booking on a Saturday gets into the schedule without any error.

In [ ]:
def is_valid(appointment):
    """Does the appointment follow the clinic's rules? Used to audit the schedule."""
    a = appointment
    try:
        day = date.fromisoformat(a["date"])
    except (TypeError, ValueError):
        return False
    return (day > TODAY and day.weekday() < 5 and a["time"] in TIMES
            and a["vet"] in VETS and a["pet"] in PETS
            and not any(c is not a and c["date"] == a["date"] and c["time"] == a["time"]
                        and c["vet"] == a["vet"] and c["status"] != "canceled"
                        for c in APPOINTMENTS))


def audit():
    """The new appointments in the schedule that do not follow the rules."""
    return [c for c in new_appointments() if not is_valid(c)]


def load_data():
    """Loads the clinic's data and resets it."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()

In [ ]:
load_data()
print(book("Max", "Fenton", "2026-10-24", "10:00", "vaccination"))
print(audit())

## 3. A more precise schema

In [ ]:
DAYS = ("Appointments Monday to Friday, from 09:00 to 13:30 and from 16:00 to 18:30, "
        "every half hour.")


PRECISE_SCHEMA = {"type": "function", "function": {
    "name": "book_appointment",
    "description": "Books an appointment at a free time. " + DAYS,
    "parameters": {"type": "object",
                   "required": ["pet", "vet", "date", "time", "reason"],
                   "properties": {
                       "pet": {"type": "string", "description": "Name of the pet."},
                       "vet": {"type": "string", "enum": []},
                       "date": {"type": "string", "format": "date",
                                "description": "Day of the appointment, YYYY-MM-DD; a "
                                               "weekday after today."},
                       "time": {"type": "string",
                                "pattern": "^(09|1[0-3]|1[6-8]):(00|30)$",
                                "description": "HH:MM, on the hour or half past."},
                       "reason": {"type": "string",
                                  "description": "Reason for the appointment."},
                       "owner": {"type": "string", "description": "Owner's name; only "
                                 "if several pets have that name."}}}}}

## 4. The contract with Pydantic

First only the types and the format; then, the clinic's rules and errors that explain how to fix them.

In [ ]:
class BookingTypes(BaseModel):
    """The booking's fields, with their types, patterns and descriptions."""
    pet: str = Field(description="Name of the pet.")
    vet: str = Field(description="Name of the vet.")
    date: datetime.date = Field(description="Day of the appointment, YYYY-MM-DD.")
    time: str = Field(pattern=r"^(09|1[0-3]|1[6-8]):(00|30)$",
                      description="HH:MM, on the hour or half past.")
    reason: str = Field(min_length=3, description="Reason for the appointment.")
    owner: str = Field("", description="Owner's name; only if several pets have that "
                                       "name.")


class Booking(BookingTypes):
    """The booking with the clinic's rules, written as validators."""

    @field_validator("vet")
    @classmethod
    def known_vet(cls, v):
        """The vet must exist; it is stored with their full name."""
        if vet_of(v) is None:
            raise ValueError(f"there is no vet \"{v}\"; they are "
                             + ", ".join(VETS.values()))
        return VETS[vet_of(v)]

    @field_validator("date")
    @classmethod
    def opening_day(cls, v):
        """The day must be in the future and from Monday to Friday."""
        if v <= TODAY:
            raise ValueError(f"{v} is not after today ({TODAY}); the booking has to be "
                             "for a future day")
        if v.weekday() >= 5:
            raise ValueError(f"{v} is a {'Saturday' if v.weekday() == 5 else 'Sunday'} "
                             "and the clinic is closed; it opens Monday to Friday")
        return v

    @field_validator("time", mode="before")
    @classmethod
    def opening_time(cls, v):
        """The time must be an appointment time; "9" or "10.30" are normalized."""
        m = re.fullmatch(r"(\d{1,2})[:.h]?(\d{2})?", str(v).strip())
        t = m and f"{int(m.group(1)):02d}:{m.group(2) or '00'}"
        if t not in TIMES:
            raise ValueError(f"\"{v}\" is not an appointment time; appointments are "
                             "on the hour or half past, from 09:00 to 13:30 and from "
                             "16:00 to 18:30")
        return t


def explain(error):
    """Pydantic's errors, written for the model: which field and how to fix it."""
    parts = []
    for e in error.errors():
        if e["type"] == "missing":
            text = "this is missing"
        elif e["type"].startswith("date"):
            text = (f"\"{e['input']}\" is not a valid date; use YYYY-MM-DD, such as "
                    "2026-10-26")
        else:
            text = e["msg"].removeprefix("Value error, ")
        parts.append(f"{e['loc'][0] if e['loc'] else 'booking'}: {text}")
    return "; ".join(parts)


def book_validated(contract, explain_errors, **arguments):
    """Validates the booking with the Pydantic model before touching the schedule.
    With explain_errors, the error says what to fix and, if the time is taken, gives
    the free times of that day; without it, it is Pydantic's error text."""
    try:
        r = contract(**arguments)
    except ValidationError as e:
        return {"error": explain(e) if explain_errors else str(e)}
    result = book(r.pet, r.vet, r.date.isoformat(), r.time, r.reason, r.owner)
    if explain_errors and "already has an appointment" in result.get("error", ""):
        result["free_times"] = free_slots(r.date.isoformat(), r.vet)["free_times"]
    return result


def schema_of(pydantic_model, name, description):
    """The tool's schema, taken from the Pydantic model."""
    parameters = pydantic_model.model_json_schema()
    parameters.pop("title", None)
    parameters.pop("description", None)
    for p in parameters["properties"].values():
        p.pop("title", None)
    return {"type": "function", "function": {"name": name, "description": description,
                                             "parameters": parameters}}

In [ ]:
for contract, explained in ((BookingTypes, False), (Booking, True)):
    for arguments in ({"date": "2026-10-24", "time": "10:00"},
                      {"date": "2026-10-26", "time": "15:00"},
                      {"date": "10/26/2026", "time": "17:30"},
                      {"date": "2026-10-26", "time": "9"}):
        reset()
        print(contract.__name__, arguments, "→",
              book_validated(contract, explained, pet="Max", vet="Fenton",
                             reason="vaccination", **arguments))

In [ ]:
print(json.dumps(schema_of(Booking, "book_appointment", "Books an appointment."),
                 ensure_ascii=False, indent=1))

## 5. The twelve bookings with the four versions

In [ ]:
def toolset(version):
    """The tools and schemas of each version; only book_appointment changes."""
    functions, schemas = dict(TASKS_SET), list(SCHEMAS_TASKS)
    new = None
    if version == "Precise schema":
        new = copy.deepcopy(PRECISE_SCHEMA)
        new["function"]["parameters"]["properties"]["vet"]["enum"] = list(VETS.values())
    elif version == "Pydantic types":
        functions["book_appointment"] = lambda **a: book_validated(BookingTypes, False,
                                                                   **a)
        new = schema_of(BookingTypes, "book_appointment",
                        "Books an appointment at a free time.")
    elif version == "Full contract":
        functions["book_appointment"] = lambda **a: book_validated(Booking, True, **a)
        new = schema_of(Booking, "book_appointment",
                        "Books an appointment at a free time. " + DAYS)
    if new:
        schemas = [new if s["function"]["name"] == "book_appointment" else s
                   for s in schemas]
    return functions, schemas


VERSIONS = ["No validation", "Precise schema", "Pydantic types", "Full contract"]


def booked(*expected):
    """Is the only new appointment the expected one (pet, vet, date, time)?"""
    return [(PETS[c["pet"]]["name"], c["vet"], c["date"], c["time"])
            for c in new_appointments()] == [expected]


def none_and_says(answer, *forms):
    """No new appointment, and the answer explains why."""
    return not new_appointments() and says(answer, *forms)


NO = (r"not possible|can't|cannot|unable|isn't (available|possible|open|valid)|"
      r"not available|closed|\btaken\b|already booked|not (a )?valid|invalid|"
      r"(doesn't|does not) exist|no vet (named|called)")


BOOKINGS = [
    ("taken_time", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd "
     "at 9.",
     lambda r: none_and_says(r, "taken", "already has", "available", NO)),
    ("quarter_past_nine", "Book a consultation for Rocky with Dr. Anna Sutton on "
     "Monday the 26th at a quarter past nine.",
     lambda r: none_and_says(r, "on the hour", "half past", "half hour", "half-hour",
                             "30 minutes", NO)),
    ("saturday", "Book a vaccination for Max with Dr. Lucy Fenton on Saturday the 24th "
     "at 10.",
     lambda r: none_and_says(r, "saturday", "weekend",
                             r"monday (to|through|-) friday")),
    ("three_pm", "Book a checkup for Kira with Dr. Paul Meyer on Monday the 26th at "
     "3 p.m.",
     lambda r: none_and_says(r, "16:00", r"\b4(:00)?\s?p\.?\s?m", "hours", NO)),
    ("september_31", "Book a checkup for Lola with Dr. Anna Sutton on September 31 "
     "at 10.",
     lambda r: none_and_says(r, "september", NO)),
    ("past", "Book wound care for Willow with Dr. Lucy Fenton on Monday the 19th "
     "at 10.",
     lambda r: none_and_says(r, r"\bpast\b", "passed", "previous", "earlier", "already",
                             NO)),
    ("slash", "Book a consultation for Simba with Dr. Paul Meyer on 10/26 at 5:30 p.m.",
     lambda r: booked("Simba", "V2", "2026-10-26", "17:30")),
    ("normal", "Book a deworming for Thor with Dr. Anna Sutton on Monday the 26th at "
     "10:30.",
     lambda r: booked("Thor", "V1", "2026-10-26", "10:30")),
    ("four_thirty", "Book a checkup for Coco with Dr. Lucy Fenton on Monday the 26th "
     "at four thirty in the afternoon.",
     lambda r: booked("Coco", "V3", "2026-10-26", "16:30")),
    ("vet_does_not_exist", "Book a consultation for Luna with Dr. Garcia on Monday the "
     "26th at 11.",
     lambda r: none_and_says(r, "Garcia", NO) or none_and_says(r, "Anna Sutton")
     and says(r, "Lucy Fenton")),
    ("first_afternoon", "Book a checkup for Bruno with Dr. Anna Sutton on Friday the "
     "23rd, at the first free time in the afternoon.",
     lambda r: booked("Bruno", "V1", "2026-10-23", "17:00")),
    ("noon", "Book a checkup for Misha with Dr. Paul Meyer on Monday the 26th at noon.",
     lambda r: booked("Misha", "V2", "2026-10-26", "12:00")),
]


def solve(functions, schemas, tasks=BOOKINGS, max_steps=8):
    """Solves each booking with the clinic reset; records the result and the audit."""
    rows = []
    for key, task, check in tasks:
        reset()
        answer, trace = agent(task, functions, schemas, max_steps)
        calls = [c for s in trace for c in s["calls"]]
        rows.append({
            "task": key, "ok": bool(check(answer)), "answer": answer,
            "steps": len(trace), "calls": [(c["name"], c["arguments"]) for c in calls],
            "rejected": sum(c["name"] == "book_appointment" and "error" in c["result"]
                            for c in calls),
            "new_appointments": new_appointments(), "invalid": audit(), "trace": trace})
    return rows


def summary(rows):
    """The totals of a version over the twelve bookings."""
    return {"ok": sum(r["ok"] for r in rows),
            "invalid_appointments": sum(len(r["invalid"]) for r in rows),
            "valid_appointments": sum(len(r["new_appointments"]) - len(r["invalid"])
                                      for r in rows),
            "rejected_bookings": sum(r["rejected"] for r in rows),
            "steps": sum(r["steps"] for r in rows)}

In [ ]:
reset()
agent(BOOKINGS[2][1], *toolset("No validation"), show=True)
print("Audit:", audit())
reset()
agent(BOOKINGS[2][1], *toolset("Full contract"), show=True)
print("Audit:", audit())

In [ ]:
results = {}
for version in VERSIONS:
    results[version] = solve(*toolset(version))
    print(f"{version:24s}", summary(results[version]))
    for r in results[version]:
        for c in r["invalid"]:
            print(f"    invalid in {r['task']}: {c['date']} {c['time']} "
                  f"{c['vet']}")

## Exercises

1. Explain in your own words the difference between a schema, a type and a business rule. Give an example of each one in the booking of an appointment.

2. Print the schema that `schema_of(Booking, …)` generates and compare it with the one of version 2. What does one have that the other does not? How many tokens does each take?

3. Create `Booking` objects by hand with these times: "9", "9:00", "09.30", "13:45", "18:30" and "nine". Which does it accept, and what does it turn them into?

4. Add a new rule to `Booking`: vaccinations are only given in the morning. What task would you write to test it? What should the error message look like?

5. Run version 3 with the explained errors, without the business rules. How much of the improvement is due to the messages and how much to the rules?

6. Make `free_slots` use the same contract as the booking, so that it does not offer times on Saturdays or in the past. Does anything change in the twelve bookings?

7. Write a contract for `notify_owner`: the text cannot be empty, longer than 300 characters or contain a phone number. Test it with three notices.

8. In the version without validation, the agent notified Willow's owner on its own, without anyone asking it to, when it saw that the time was not free. Which part of the system should prevent those unrequested actions: the system message, the tool or the loop?

9. Try the version with the full contract with Qwen3-1.7B, without the "Thought:" line of the system message. Does any invalid appointment reach the schedule? Why not?

10. **Project.** Write the full contract of `cancel_appointment`: the appointment has to exist, be scheduled and not be today's (today's are canceled by phone). Design five tasks, measure the tool with and without the contract and write a short report.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). Pydantic's documentation on [validators](https://docs.pydantic.dev/latest/concepts/validators/) and [JSON schemas](https://docs.pydantic.dev/latest/concepts/json_schema/). [JSON Schema](https://json-schema.org/understanding-json-schema/). Northside Veterinary Clinic and its data are fictional. Original code and experiments.